# Lecture 1 — First-order formulas and Prover9

This notebook follows the exercises of Lecture 1, step by step. Keep the slides open next to it
(`slides/en/lecture1.pdf`).

How to use it:

- run the cells **from top to bottom** (Shift+Enter);
- the cells marked ✏️ are for you: replace every `???`, then run the cell to save your file
  (`%%writefile` writes the cell into a file of the folder `work/`), then run the next cell;
- **predict, then run**: before each run, write your prediction in the Markdown cell above it.

The notebook uses the helper `p9.py` of this folder. Nothing is sent outside your machine.

In [2]:
# Run this cell first. PY is the Python of this notebook: the commands below use it.
import os, sys, shutil, subprocess, pathlib
PY = f'"{sys.executable}"'
os.makedirs('work', exist_ok=True)          # the files you write go to work/
print('folder:', os.getcwd())

folder: /home/user/cours/DKAI/neursymbolic/Lab1/Lab1/exercises/lecture1


In [3]:
# A small helper: run Prover9 on a file exactly as on the command line (prover9 -f FILE),
# then show its exit code, the proof and the error message, if any.
import p9
P9 = p9.find_prover9()

def prover9(path):
    done = subprocess.run([P9, '-f', path], capture_output=True, text=True)
    print('exit code', done.returncode, '=', p9.EXIT_CODES.get(done.returncode, '?'))
    for line in p9.proof_lines(done.stdout):
        print('   ', line)
    if p9.error_message(done.stdout):
        print('   ', p9.error_message(done.stdout))
    return done.stdout

def section(output, title):
    """One section of the output of Prover9, for example 'PROCESS INITIAL CLAUSES'."""
    part = output.split(title, 1)[1].split('=====', 1)[1] if title in output else ''
    print(part.split('end of', 1)[0].strip())

## 1. Check the installation

The three lines must end without error (slides, part 1).

In [4]:
!{PY} p9.py check

Prover9 program: /home/user/cours/DKAI/neursymbolic/Lab1/Lab1/.venv/bin/prover9
Version:         Prover9 (64) version 2009-11A, November 2009.
Socrates test:   exit code 0 -> OK


## 2. A first run: Socrates

The input file, then one run, then the label (two runs).

In [5]:
print(open('files/socrates.in', encoding='utf-8').read())

% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(socrates).
end_of_list.



In [6]:
out = prover9('files/socrates.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all x (man(x) -> mortal(x))) # label(non_clause).  [assumption].
    2 mortal(socrates) # label(non_clause) # label(goal).  [goal].
    3 man(socrates).  [assumption].
    4 -man(x) | mortal(x).  [clausify(1)].
    5 mortal(socrates).  [resolve(3,a,4,a)].
    6 -mortal(socrates).  [deny(2)].
    7 $F.  [resolve(5,a,6,a)].


In [7]:
!{PY} p9.py label files/socrates.in

files/socrates.in: goal -> code 0, -(goal) -> code 2  =>  True  (0.01 s)


A search that does not finish in 2 seconds: exit code 4, *we do not know*.

In [8]:
out = prover9('files/demo_timeout.in')

exit code 4 = SEARCH FAILED, max_seconds (time limit reached: we do not know)


The worked example of part 3 (FOLIO, example 240):

In [9]:
print(open('files/example_lewandowski.in', encoding='utf-8').read())
!{PY} p9.py label files/example_lewandowski.in

% example_lewandowski.in -- the worked example of part 3 (complete: run it as it is)
% FOLIO validation, story 79, example 240
%   Robert Lewandowski is a striker.  Strikers are soccer players.
%   Robert Lewandowski left Bayern Munchen.
%   If a player leaves a team they no longer play for that team.
%   Conclusion: Robert Lewandowski is a soccer player.

formulas(assumptions).
  striker(lewandowski).                                     % R. Lewandowski is a striker.
  all x (striker(x) -> soccer_player(x)).                   % Strikers are soccer players.
  left(lewandowski, bayern).                                % R. Lewandowski left Bayern Munchen.
  all x all t ((soccer_player(x) & left(x, t)) -> -plays_for(x, t)).
                                                            % If a player leaves a team ...
end_of_list.

formulas(goals).
  soccer_player(lewandowski).                               % R. Lewandowski is a soccer player.
end_of_list.

files/example_lewandowski.in: goal 

## Exercise 1 — sentences to formulas (easy)

✏️ Write one Prover9 formula after each `=` (the vocabulary is in the comments), run the cell,
then run the checker. `too strong`: your formula says more than the sentence; `too weak`: less.

In [14]:
%%writefile work/ex1_formulas.txt
% Exercise 1 -- English sentences as first-order formulas, in Prover9 syntax.
%
% Write ONE Prover9 formula after each "=" (the final period is optional), then run:
%     python check_formulas.py
%
% Vocabulary (use nothing else; names are case-sensitive):
%     predicates   student(x)  smart(x)  lazy(x)  passes(x)  knows(x, y)    "x knows y"
%     function     advisor(x)                                                "the advisor of x"
%     constants    alice  bob
%
% Reminder:   -  not    &  and    |  or    ->  implies    <->  if and only if
%             =  equal  !=  different
%             all x (...)     exists x (...)
%
% Example (already done):  "Alice is a student."
s0 = student(alice)

% s1.  Bob is not lazy.
s1 = -lazy(bob)

% s2.  All students are smart.
s2 = all x (student(x) -> smart(x))

% s3.  Some students are lazy.
s3 = exists x (student(x) & lazy(x))

% s4.  No lazy student passes.
s4 = all x (lazy(x) & student(x) -> -passes(x))

% s5.  Not every student passes.
s5 = exists x (student(x) & -passes(x))

% s6.  Alice is smart or lazy, but not both.
s6 = (smart(alice) | lazy(alice)) & -(smart(alice) & lazy(alice))

% s7.  Bob passes only if Alice passes.
s7 = passes(bob) -> passes(alice)

% s8.  Alice passes unless she is lazy.
s8 = -lazy(alice) -> passes(alice)

% s9.  Only students pass.
s9 = all x (-student(x) -> -passes(x))

% s10. Every student knows some smart student.
s10 = all x exists y (student(x) -> knows(x, y) & student(y) & smart(y))

% s11. There is someone whom every student knows.
s11 = exists y all x (student(x) -> knows(x, y))

% s12. The advisor of every student is smart.
s12 = all x (student(x) -> smart(advisor(x)))

% s13. Alice and Bob are two different students.
s13 = student(alice) & student(bob) & alice != bob

% s14. At most one student is lazy.
s14 = all x (student(x) & lazy(x) -> all y ((student(y) & lazy(y)) -> y = x))

Overwriting work/ex1_formulas.txt


In [15]:
!{PY} check_formulas.py work/ex1_formulas.txt

  [ok] s1
  [ok] s2
  [ok] s3
  [ok] s4
  [ok] s5
  [ok] s6
  [ok] s7
  [ok] s8
  [ok] s9
  [ok] s10
  [ok] s11
  [ok] s12
  [ok] s13
  [ok] s14
SUCCESS


## Exercise 2 — predict, then run (easy)

Each cell below starts from `socrates.in`. ✏️ Make the change written in its first line, **write
your prediction** (exit code 0, 1 or 2) in the Markdown cell, then run both cells.

**(a)** delete the premise man(socrates).  
My prediction: code 2 ? You can't deduce the mortality of socrates from that without knowing he's a man

In [19]:
%%writefile work/ex2a.in
% (a) delete the premise man(socrates).
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
%  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(socrates).
end_of_list.

Overwriting work/ex2a.in


In [20]:
out = prover9('work/ex2a.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


**(b)** delete the period after man(socrates)  
My prediction: should not change anything, it's optional

In [21]:
%%writefile work/ex2b.in
% (b) delete the period after man(socrates)
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates)
end_of_list.

% The question (phi)
formulas(goals).
  mortal(socrates).
end_of_list.

Writing work/ex2b.in


In [22]:
out = prover9('work/ex2b.in')

exit code 1 = ERROR (the input file is not valid: syntax, arity, ...)
    A term cannot be constructed from the marked string: >>> man(socrates) end_of_list. <<<


Ah it's not

**(c)** write man(Socrates). (capital S)  
My prediction: socrates and Socrates are two different variables so the results is the same as in (a)

In [27]:
%%writefile work/ex2c.in
% (c) write man(Socrates). (capital S)
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(Socrates).
end_of_list.

Overwriting work/ex2c.in


In [28]:
out = prover9('work/ex2c.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


**(d)** ask mortal(socrates, athens).  
My prediction: Error of arrity ?

In [23]:
%%writefile work/ex2d.in
% (d) ask mortal(socrates, athens).
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(socrates, athens).
end_of_list.

Writing work/ex2d.in


In [24]:
out = prover9('work/ex2d.in')

exit code 1 = ERROR (the input file is not valid: syntax, arity, ...)
    The following symbols are used with multiple arities: mortal/1, mortal/2.


**(e)** replace man(socrates). by man(xavier). and ask mortal(plato).  
My prediction: Code 2 end of search etc. We can't deduce anything here, just as with Socrates and socrates

In [29]:
%%writefile work/ex2e.in
% (e) replace man(socrates). by man(xavier). and ask mortal(plato).
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(xavier).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(plato).
end_of_list.

Writing work/ex2e.in


In [30]:
out = prover9('work/ex2e.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all x (man(x) -> mortal(x))) # label(non_clause).  [assumption].
    2 mortal(plato) # label(non_clause) # label(goal).  [goal].
    3 man(x).  [assumption].
    4 -man(x) | mortal(x).  [clausify(1)].
    5 mortal(x).  [resolve(3,a,4,a)].
    6 -mortal(plato).  [deny(2)].
    7 $F.  [resolve(5,a,6,a)].


WTF ???

Ah xavier is considered a universal quantifier...................................................... Because it starts with an x...................................................

**(f)** add a second goal mortal(plato).  
My prediction: Code 2 end of search ? We can't deduce all phis

In [31]:
%%writefile work/ex2f.in
% (f) add a second goal mortal(plato).
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(socrates).
  mortal(plato).
end_of_list.

Writing work/ex2f.in


In [32]:
out = prover9('work/ex2f.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all x (man(x) -> mortal(x))) # label(non_clause).  [assumption].
    2 mortal(socrates) # label(non_clause) # label(goal).  [goal].
    4 man(socrates).  [assumption].
    5 -man(x) | mortal(x).  [clausify(1)].
    6 mortal(socrates).  [resolve(4,a,5,a)].
    7 -mortal(socrates).  [deny(2)].
    9 $F.  [resolve(6,a,7,a)].


It's.... OR ?

**(g)** replace the goal by mortal(x).  
My prediction: So x -> universal quantifier. So we're tryna prove that everything is mortal. With our premises, we can't derive that so code 2 seems reasonable

In [33]:
%%writefile work/ex2g.in
% (g) replace the goal by mortal(x).
% socrates.in -- the first example of Lecture 1 (complete: run it as it is)
%   All men are mortal.  Socrates is a man.  Is Socrates mortal?

% The premises (K)
formulas(assumptions).
  all x (man(x) -> mortal(x)).
  man(socrates).
end_of_list.

% The question (phi)
formulas(goals).
  mortal(x).
end_of_list.

Writing work/ex2g.in


In [34]:
out = prover9('work/ex2g.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


For (e), look at the `[assumption]` line of `man(...)`; for (g), the `[deny(2)]` line.

## Exercise 3 — "all" and "some" (medium)

(a) ✏️ Translate, run, and find the line of the proof that introduces `c1`.

In [35]:
%%writefile work/ex3a.in
% Exercise 3 -- "all" and "some"
%   All cats are mammals.  Some cats are black.  Are some mammals black?
% Predicates: cat/1, mammal/1, black/1.   Replace every ??? and run.

formulas(assumptions).
  all x (cat(x) -> mammal(x)).          % All cats are mammals.
  exists x (cat(x) & black(x)).          % Some cats are black.
end_of_list.

formulas(goals).
  exists x (mammal(x) & black(x)).          % Some mammals are black.
end_of_list.

Writing work/ex3a.in


In [36]:
out = prover9('work/ex3a.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all x (cat(x) -> mammal(x))) # label(non_clause).  [assumption].
    2 (exists x (cat(x) & black(x))) # label(non_clause).  [assumption].
    3 (exists x (mammal(x) & black(x))) # label(non_clause) # label(goal).  [goal].
    4 cat(c1).  [clausify(2)].
    5 -cat(x) | mammal(x).  [clausify(1)].
    6 -mammal(x) | -black(x).  [deny(3)].
    7 black(c1).  [clausify(2)].
    8 -mammal(c1).  [resolve(6,b,7,a)].
    9 mammal(c1).  [resolve(4,a,5,a)].
    10 $F.  [resolve(8,a,9,a)].


(b) ✏️ The second premise suggested by a language model: `exists x (cat(x) -> black(x))`.

In [41]:
%%writefile work/ex3b.in
% Exercise 3 -- "all" and "some"
%   All cats are mammals.  Some cats are black.  Are some mammals black?
% Predicates: cat/1, mammal/1, black/1.   Replace every ??? and run.

formulas(assumptions).
  all x (cat(x) -> mammal(x)).          % All cats are mammals.
  exists x (cat(x) -> black(x)).          % Some cats are black.
end_of_list.

formulas(goals).
  exists x (mammal(x) & black(x)).          % Some mammals are black.
end_of_list.

Overwriting work/ex3b.in


In [38]:
out = prover9('work/ex3b.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


(c) ✏️ "All cats are black" instead of "some cats are black".

In [42]:
%%writefile work/ex3c.in
% Exercise 3 -- "all" and "some"
%   All cats are mammals.  Some cats are black.  Are some mammals black?
% Predicates: cat/1, mammal/1, black/1.   Replace every ??? and run.

formulas(assumptions).
  all x (cat(x) -> mammal(x)).          % All cats are mammals.
  all x (cat(x) -> black(x)).          % All cats are black.
end_of_list.

formulas(goals).
  exists x (mammal(x) & black(x)).          % Some mammals are black.
end_of_list.

Writing work/ex3c.in


In [43]:
out = prover9('work/ex3c.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


## Exercise 4 — the order of quantifiers (medium)

✏️ (a) assumption R2, goal R1; (b) assumption R1, goal R2.

In [46]:
%%writefile work/ex4a.in
% Exercise 4 -- the order of quantifiers
%   "Every student has read a book"
%   R1 (each their own book):  for every student x there is a book y that x has read
%   R2 (one same book):        there is a book y that every student x has read
% Predicates: student/1, book/1, read/2.
% Make two copies of this file:
%   ex4a.in   assumption R2, goal R1
%   ex4b.in   assumption R1, goal R2

formulas(assumptions).
  exists y all x (student(x) -> book(y) & read(x, y)).
end_of_list.

formulas(goals).
  all x exists y (student(x) -> book(y) & read(x, y)).
end_of_list.

Overwriting work/ex4a.in


In [47]:
out = prover9('work/ex4a.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (exists y all x (student(x) -> book(y) & read(x,y))) # label(non_clause).  [assumption].
    2 (all x exists y (student(x) -> book(y) & read(x,y))) # label(non_clause) # label(goal).  [goal].
    3 student(c2).  [deny(2)].
    4 -student(x) | book(c1).  [clausify(1)].
    5 -student(x) | read(x,c1).  [clausify(1)].
    6 book(c1).  [resolve(3,a,4,a)].
    7 -book(x) | -read(c2,x).  [deny(2)].
    8 -read(c2,c1).  [resolve(6,a,7,a)].
    9 read(c2,c1).  [resolve(3,a,5,a)].
    10 $F.  [resolve(8,a,9,a)].


In [48]:
%%writefile work/ex4b.in
% Exercise 4 -- the order of quantifiers
%   "Every student has read a book"
%   R1 (each their own book):  for every student x there is a book y that x has read
%   R2 (one same book):        there is a book y that every student x has read
% Predicates: student/1, book/1, read/2.
% Make two copies of this file:
%   ex4a.in   assumption R2, goal R1
%   ex4b.in   assumption R1, goal R2

formulas(assumptions).
  all x exists y (student(x) -> book(y) & read(x, y)).
end_of_list.

formulas(goals).
  exists y all x (student(x) -> book(y) & read(x, y)).
end_of_list.

Writing work/ex4b.in


In [49]:
out = prover9('work/ex4b.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


(d) The clauses of (b): find `f1(x)`. What does it stand for?

It's a function that gather the book which x has read. Ie: "Il existe f1(x) un livre lu par x"

In [50]:
section(out, 'PROCESS INITIAL CLAUSES')


% Clauses before input processing:

formulas(usable).
end_of_list.

formulas(sos).
-student(x) | book(f1(x)).  [clausify(1)].
-student(x) | read(x,f1(x)).  [clausify(1)].
student(f2(x)).  [deny(2)].
-book(x) | -read(f2(x),x).  [deny(2)].
end_of_list.

formulas(demodulators).
end_of_list.

============================== PREDICATE ELIMINATION =================

Eliminating student/1
3 student(f2(x)).  [deny(2)].
4 -student(x) | book(f1(x)).  [clausify(1)].
5 -student(x) | read(x,f1(x)).  [clausify(1)].
Derived: book(f1(f2(x))).  [resolve(3,a,4,a)].
Derived: read(f2(x),f1(f2(x))).  [resolve(3,a,5,a)].

Eliminating book/1
6 book(f1(f2(x))).  [resolve(3,a,4,a)].
7 -book(x) | -read(f2(x),x).  [deny(2)].
Derived: -read(f2(f1(f2(x))),f1(f2(x))).  [resolve(6,a,7,a)].

Eliminating read/2
8 -read(f2(f1(f2(x))),f1(f2(x))).  [resolve(6,a,7,a)].
9 read(f2(x),f1(f2(x))).  [resolve(3,a,5,a)].

============================== end predicate elimination =============

Auto_denials:  (no changes).

Term o

## Exercise 5 — the galette des rois (hard)

(a) ✏️ Translate the 6 sentences with the given vocabulary, then prove that Victor is crowned.
(b) Which sentence produced `c1`?

In [53]:
%%writefile work/ex5a.in
% Exercise 5 -- the galette des rois
% Vocabulary (do not change it):
%   gives(x, s, p)    x gives slice s to p        colleague(p, q)   p is a colleague of q
%   youngest(x)  slice(s)  has_feve(s)  guest(p)  crowned(p)
%   constants: chloe, lea, p_victor

formulas(assumptions).
  all y all x all s (guest(x) & slice(s) & has_feve(s) & gives(y, s, x) & youngest(y) -> crowned(x)).   % (1) A guest who is given a slice containing the feve by the youngest child is crowned.
  (all s1 all s2 (slice(s1) & slice(s2) & has_feve(s1) & has_feve(s2) -> s1 = s2)) & exists s (slice(s) & has_feve(s)).   % (2) One slice of the galette contains the feve.
  all s (slice(s) & has_feve(s) -> gives(chloe, s, p_victor)).   % (3) Chloe gave every slice containing the feve to Victor.
  youngest(chloe).   % (4) Chloe is the youngest child.
  all c (colleague(c, lea) -> guest(c)).   % (5) Every colleague of Lea is a guest.
  colleague(p_victor, lea).   % (6) Victor is a colleague of Lea.
end_of_list.

formulas(goals).
  crowned(p_victor).   % Is Victor crowned?
end_of_list.

Overwriting work/ex5a.in


In [54]:
out = prover9('work/ex5a.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all y all x all s (guest(x) & slice(s) & has_feve(s) & gives(y,s,x) & youngest(y) -> crowned(x))) # label(non_clause).  [assumption].
    2 (all s1 all s2 (slice(s1) & slice(s2) & has_feve(s1) & has_feve(s2) -> s1 = s2)) & (exists s (slice(s) & has_feve(s))) # label(non_clause).  [assumption].
    3 (all s (slice(s) & has_feve(s) -> gives(chloe,s,p_victor))) # label(non_clause).  [assumption].
    4 (all c (colleague(c,lea) -> guest(c))) # label(non_clause).  [assumption].
    5 crowned(p_victor) # label(non_clause) # label(goal).  [goal].
    6 -colleague(x,lea) | guest(x).  [clausify(4)].
    7 -guest(x) | -slice(y) | -has_feve(y) | -gives(z,y,x) | -youngest(z) | crowned(x).  [clausify(1)].
    8 -colleague(x,lea) | -slice(y) | -has_feve(y) | -gives(z,y,x) | -youngest(z) | crowned(x).  [resolve(6,b,7,a)].
    9 -slice(x) | -has_feve(x) | gives(chloe,x,p_victor).  [clausify(3)].
    11 slice(c1).  [clausify(2)].
    12 has_feve(c

(c) ✏️ Copy your translation and ask: is Lea crowned?

In [59]:
%%writefile work/ex5c.in
% Exercise 5 -- the galette des rois
% Vocabulary (do not change it):
%   gives(x, s, p)    x gives slice s to p        colleague(p, q)   p is a colleague of q
%   youngest(x)  slice(s)  has_feve(s)  guest(p)  crowned(p)
%   constants: chloe, lea, p_victor

formulas(assumptions).
  all y all x all s (guest(x) & slice(s) & has_feve(s) & gives(y, s, x) & youngest(y) -> crowned(x)).   % (1) A guest who is given a slice containing the feve by the youngest child is crowned.
  (all s1 all s2 (slice(s1) & slice(s2) & has_feve(s1) & has_feve(s2) -> s1 = s2)) & exists s (slice(s) & has_feve(s)).   % (2) One slice of the galette contains the feve.
  all s (slice(s) & has_feve(s) -> gives(chloe, s, p_victor)).   % (3) Chloe gave every slice containing the feve to Victor.
  youngest(chloe).   % (4) Chloe is the youngest child.
  all c (colleague(c, lea) -> guest(c)).   % (5) Every colleague of Lea is a guest.
  colleague(p_victor, lea).   % (6) Victor is a colleague of Lea.
end_of_list.

formulas(goals).
  crowned(lea).
end_of_list.

Overwriting work/ex5c.in


In [60]:
out = prover9('work/ex5c.in')

exit code 2 = SEARCH FAILED, sos_empty (the search is finished: no proof from these assumptions)


(d) ✏️ Replace `p_victor` by `victor` everywhere, then ask (c) again.

In [61]:
%%writefile work/ex5d.in
% Exercise 5 -- the galette des rois
% Vocabulary (do not change it):
%   gives(x, s, p)    x gives slice s to p        colleague(p, q)   p is a colleague of q
%   youngest(x)  slice(s)  has_feve(s)  guest(p)  crowned(p)
%   constants: chloe, lea, p_victor

formulas(assumptions).
  all y all x all s (guest(x) & slice(s) & has_feve(s) & gives(y, s, x) & youngest(y) -> crowned(x)).   % (1) A guest who is given a slice containing the feve by the youngest child is crowned.
  (all s1 all s2 (slice(s1) & slice(s2) & has_feve(s1) & has_feve(s2) -> s1 = s2)) & exists s (slice(s) & has_feve(s)).   % (2) One slice of the galette contains the feve.
  all s (slice(s) & has_feve(s) -> gives(chloe, s, victor)).   % (3) Chloe gave every slice containing the feve to Victor.
  youngest(chloe).   % (4) Chloe is the youngest child.
  all c (colleague(c, lea) -> guest(c)).   % (5) Every colleague of Lea is a guest.
  colleague(victor, lea).   % (6) Victor is a colleague of Lea.
end_of_list.

formulas(goals).
  crowned(lea).
end_of_list.

Writing work/ex5d.in


In [62]:
out = prover9('work/ex5d.in')

exit code 0 = THEOREM PROVED (a proof was found)
    1 (all y all x all s (guest(x) & slice(s) & has_feve(s) & gives(y,s,x) & youngest(y) -> crowned(x))) # label(non_clause).  [assumption].
    2 (all s1 all s2 (slice(s1) & slice(s2) & has_feve(s1) & has_feve(s2) -> s1 = s2)) & (exists s (slice(s) & has_feve(s))) # label(non_clause).  [assumption].
    3 (all s (slice(s) & has_feve(s) -> gives(chloe,s,victor))) # label(non_clause).  [assumption].
    4 (all c (colleague(c,lea) -> guest(c))) # label(non_clause).  [assumption].
    5 crowned(lea) # label(non_clause) # label(goal).  [goal].
    6 -colleague(x,lea) | guest(x).  [clausify(4)].
    7 -guest(x) | -slice(y) | -has_feve(y) | -gives(z,y,x) | -youngest(z) | crowned(x).  [clausify(1)].
    8 -colleague(x,lea) | -slice(y) | -has_feve(y) | -gives(z,y,x) | -youngest(z) | crowned(x).  [resolve(6,b,7,a)].
    9 -slice(x) | -has_feve(x) | gives(chloe,x,y).  [clausify(3)].
    11 slice(c1).  [clausify(2)].
    12 has_feve(c1).  [clausify

## Exercise 6 — a small experiment on FOLIO (medium)

The 10 problems are copied to `work/ex6/` (your copy). (a) Read them and predict each label.

In [63]:
if not os.path.exists('work/ex6'):
    shutil.copytree('files/ex6_experiment/problems', 'work/ex6')
for name in sorted(os.listdir('work/ex6')):
    print('=====', name)
    print(open('work/ex6/' + name, encoding='utf-8').read())

===== p01_hawks.in
% FOLIO validation, story 292, example 736
%   A hawk never swims.  Some birds are hawks.
%   Conclusion: All birds swim.
formulas(assumptions).
  all x (hawk(x) -> -swims(x)).          % A hawk never swims.
  exists x (bird(x) & hawk(x)).          % Some birds are hawks.
end_of_list.

formulas(goals).
  all x (bird(x) -> swims(x)).           % All birds swim.
end_of_list.

===== p02_greeks.in
% FOLIO validation, story 258, example 702.  Translate, replacing every ???
%   All humans are mortal.  All Greeks are humans.
%   Conclusion: Some Greeks are mortal.
formulas(assumptions).
  ???.     % All humans are mortal.
  ???.     % All Greeks are humans.
end_of_list.

formulas(goals).
  ???.     % Some Greeks are mortal.
end_of_list.

===== p03_horses.in
% FOLIO validation, story 259, example 703
%   All horses have hooves.  No humans have hooves.
%   Conclusion: Some humans are horses.
formulas(assumptions).
  all x (horse(x) -> has_hooves(x)).     % All horses have hoo

(b) ✏️ Translate the five files that contain `???`.

In [64]:
%%writefile work/ex6/p02_greeks.in
% FOLIO validation, story 258, example 702.  Translate, replacing every ???
%   All humans are mortal.  All Greeks are humans.
%   Conclusion: Some Greeks are mortal.
formulas(assumptions).
  all x (human(x) -> motal(x)).     % All humans are mortal.
  all g (greek(g) -> mortal(g)).     % All Greeks are humans.
end_of_list.

formulas(goals).
  exists g (greek(g) & mortal(g)).     % Some Greeks are mortal.
end_of_list.

Overwriting work/ex6/p02_greeks.in


In [73]:
%%writefile work/ex6/p04_events.in
% FOLIO validation, story 280, example 724.  Translate, replacing every ???
%   Events are either happy or sad.  At least one event is happy.
%   Conclusion: All events are sad.
formulas(assumptions).
  all e (event(e) -> ((happy(e) | sad(e)) & -(happy(e) & sad(e)))).     % Events are either happy or sad.
  exists e (event(e) & happy(e)).     % At least one event is happy.
end_of_list.

formulas(goals).
  all e (event(e) -> sad(e)).     % All events are sad.
end_of_list.

Overwriting work/ex6/p04_events.in


In [67]:
%%writefile work/ex6/p06_pets.in
% FOLIO validation, story 256, example 700.  Translate, replacing every ???
%   All cats are mammals.  Some pets are not mammals.
%   Conclusion: No pets are cats.
formulas(assumptions).
  all c (cat(c) -> mammal(c)).     % All cats are mammals.
  exists p (pet(p) & -mammal(p)).     % Some pets are not mammals.
end_of_list.

formulas(goals).
  all p (pet(p) -> -cat(p)).     % No pets are cats.
end_of_list.

Overwriting work/ex6/p06_pets.in


In [77]:
%%writefile work/ex6/p08_tom.in
% FOLIO validation, story 441, example 1269.  Translate, replacing every ???
%   No one nice to animals is also mean to animals.  Some grumpy people are mean to animals.
%   All animal lovers are nice to animals.  All pet owners love animals.  Tom is a pet owner.
%   Conclusion: Tom is not both a grumpy person and mean to animals.
formulas(assumptions).
  all p (nice(p) -> -mean(p)).     % No one nice to animals is also mean to animals.
  exists p (grumpy(p) & mean(p)).     % Some grumpy people are mean to animals.
  all p (lover(p) -> nice(p)).     % All animal lovers are nice to animals.
  all p (owner(p) -> lover(p)).     % All pet owners love animals.
  owner(tom).     % Tom is a pet owner.
end_of_list.

formulas(goals).
  -(grumpy(tom) & mean(tom)) .     % Tom is not both a grumpy person and mean to animals.
end_of_list.

Overwriting work/ex6/p08_tom.in


In [85]:
%%writefile work/ex6/p09_tom.in
% FOLIO validation, story 441, example 1270.  Same premises as p08_tom.in: copy them.
%   Conclusion: If Tom is not both a grumpy person and mean to animals,
%               then Tom is neither a grumpy person nor an animal lover.
formulas(assumptions).
  all p (nice(p) -> -mean(p)).     % No one nice to animals is also mean to animals.
  exists p (grumpy(p) & mean(p)).     % Some grumpy people are mean to animals.
  all p (lover(p) -> nice(p)).     % All animal lovers are nice to animals.
  all p (owner(p) -> lover(p)).     % All pet owners love animals.
  owner(tom).     % Tom is a pet owner.
end_of_list.

formulas(goals).
  -(grumpy(tom) & mean(tom)) -> (-grumpy(tom) & -lover(tom)).
end_of_list.

Overwriting work/ex6/p09_tom.in


In [86]:
!{PY} p9.py experiment work/ex6 --gold files/ex6_experiment/gold.csv --out work/results.csv

file                         goal -goal  label        gold       seconds
p01_hawks.in                    2     0  False        False         0.01 ok
p02_greeks.in                   2     2  Uncertain    Uncertain     0.01 ok
p03_horses.in                   2     0  False        False         0.01 ok
p04_events.in                   2     0  False        False         0.01 ok
p05_squares.in                  0     2  True         True          0.01 ok
p06_pets.in                     2     2  Uncertain    Uncertain     0.02 ok
p07_ted.in                      0     2  True         True          0.04 ok
p08_tom.in                      0     2  True         True          0.02 ok
p09_tom.in                      2     0  False        False         0.01 ok
p10_susan.in                    2     0  False        False         0.01 ok

agreement with gold: 10/10
results written to work/results.csv


(d) ✏️ `p04` with an **inclusive** `|`: which label?

In [ ]:
%%writefile work/p04_inclusive.in
% FOLIO validation, story 280, example 724.  Translate, replacing every ???
%   Events are either happy or sad.  At least one event is happy.
%   Conclusion: All events are sad.
formulas(assumptions).
  ???.     % Events are either happy or sad.
  ???.     % At least one event is happy.
end_of_list.

formulas(goals).
  ???.     % All events are sad.
end_of_list.

In [ ]:
!{PY} p9.py label work/p04_inclusive.in

(e) The **negation** of `p09`, run by hand: exit code? Is there a proof in the output?

In [ ]:
with open('work/p09_negated.in', 'w', encoding='utf-8') as f:
    f.write(p9.negated(open('work/ex6/p09_tom.in', encoding='utf-8').read()))
out = prover9('work/p09_negated.in')
print('proof in the output:', '= PROOF =' in out)
print([line.strip() for line in out.splitlines() if 'max_proofs' in line or 'proofs=' in line])

## Bonus — is a text consistent?

✏️ 1. The three premises, **without** a goals list.

In [ ]:
%%writefile work/bonus.in
% Bonus -- is this text consistent?
%   Every employee has access to the server.  Marc is an employee.
%   Marc does not have access to the server.
% Write the three premises. There is NO goals list in this file: run it as it is.

formulas(assumptions).
  ???.
  ???.
  ???.
end_of_list.

In [ ]:
out = prover9('work/bonus.in')

2. ✏️ Without the third premise.

In [ ]:
%%writefile work/bonus2.in
% Bonus -- is this text consistent?
%   Every employee has access to the server.  Marc is an employee.
%   Marc does not have access to the server.
% Write the three premises. There is NO goals list in this file: run it as it is.

formulas(assumptions).
  ???.
  ???.
  ???.
end_of_list.

In [ ]:
out = prover9('work/bonus2.in')

3. ✏️ The three premises and the goal `marc_is_the_pope.` — by hand, then with `p9.py`.

In [ ]:
%%writefile work/bonus3.in
% Bonus -- is this text consistent?
%   Every employee has access to the server.  Marc is an employee.
%   Marc does not have access to the server.
% Write the three premises. There is NO goals list in this file: run it as it is.

formulas(assumptions).
  ???.
  ???.
  ???.
end_of_list.

In [ ]:
out = prover9('work/bonus3.in')
!{PY} p9.py run work/bonus3.in